# Stage 2 + 3 - agent fusion search, then the one-time test evaluation

**Settings:** Accelerator **GPU T4 x2** (the local Qwen2.5-7B search policy needs both GPUs),
Internet **On**, input dataset `clwd-embeddings` attached, Secret `WANDB_API_KEY` attached
for the training/validation curves. Use **Save & Run All (Commit)** so nothing is lost.

* Stage 2 sees development patients only (train + val); the test split is never loaded.
* Stage 3 trains every results-table row with 5 seeds and touches test **once**
  (`artifacts/results/TEST_LOCK.json` prevents silent re-runs).

In [ ]:
# Clone the pinned branch of the repository and install only what Kaggle lacks.
# NOTE: no "pip -U": upgrading huggingface_hub to 2.x breaks Kaggle's transformers.
import os, subprocess, sys
REPO_DIR = "/kaggle/working/pathology-fusion-AI"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", "main", "https://github.com/jkaral/pathology-fusion-AI.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
!pip install -q -r requirements-kaggle.txt
!git log -1 --format="commit %h %s"

In [ ]:
# Secrets live in Kaggle "Add-ons -> Secrets" (never in the repo or notebook).
import os
try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    for name in ("HF_TOKEN", "WANDB_API_KEY"):
        try:
            os.environ[name] = s.get_secret(name)
            print(f"{name}: loaded")
        except Exception:
            print(f"{name}: not set")
except ImportError:
    print("Not on Kaggle: set HF_TOKEN / WANDB_API_KEY in the environment yourself.")
import torch
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
import glob
cands = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/WSI-*.h5", recursive=True)})
print(cands)
EMB = "/kaggle/working/all_embeddings"
os.makedirs(EMB, exist_ok=True)
for d in cands:                    # merge shards into one directory of symlinks
    for p in glob.glob(f"{d}/WSI-*.h5"):
        dst = os.path.join(EMB, os.path.basename(p))
        if not os.path.exists(dst):
            os.symlink(p, dst)
print(len(glob.glob(f"{EMB}/*.h5")), "feature files")

## Stage 2 - agent search (proposer `auto` = local Qwen on GPU, deterministic fallback)

In [ ]:
!python scripts/02_run_agent.py --embeddings-dir {EMB}
!cat artifacts/agent/$(cat artifacts/agent/LATEST)/decisions.md
# shortcut check (dev OOF only): does performance survive within slide-ID acquisition blocks?
!python scripts/08_within_batch_eval.py --run $(cat artifacts/agent/LATEST)

## Stage 3 - final evaluation (runs once)

In [ ]:
!python scripts/03_final_evaluation.py --embeddings-dir {EMB}

In [ ]:
# Bundle the small, committable artifacts (no features) for download -> add to the repo.
!tar czf /kaggle/working/results_bundle.tgz artifacts/agent artifacts/results artifacts/within_batch
print("Download /kaggle/working/results_bundle.tgz from the Output tab")